# SC2079 symbol card detection (YOLO26)

Fine-tunes a pretrained YOLO26-nano detector to localize and classify the 30 symbol cards (image IDs 11–40).

Pipeline:
1. Capture images with `photobooth.py` into `dataset/{train,test}/<id>_<slug>/` (auto-labels each shot with a YOLO sidecar).
2. Run `autolabel.py` to build `dataset_yolo/` (images + labels + `data.yaml`), then skim `dataset_yolo/review/` for bad boxes.
3. Train, validate and export below.

Setup: `uv pip install -p .venv ultralytics` (pulls in torch, opencv, matplotlib).

In [1]:
import random
import sys
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device

'cuda'

In [2]:
ROOT = Path.cwd()
YOLO_DATASET = ROOT / "dataset_yolo"
DATA_YAML = YOLO_DATASET / "data.yaml"

# rebuild labels + data.yaml from the ImageFolder captures in dataset/
# (skips already-labeled images; use --force to redo everything)
!{sys.executable} autolabel.py

train/ -> train/


  labeled 0 (sidecar 0), skipped 281, failed 1
test/ -> val/


  labeled 0 (sidecar 0), skipped 70, failed 0


Wrote /workspace/image-rec/dataset_yolo/data.yaml
Review images (boxes drawn) in /workspace/image-rec/dataset_yolo/review

1 images could not be auto-labeled:
  /workspace/image-rec/dataset/train/12_2/12_0005.jpg
Label these by hand or recapture them.


### Dataset sanity check

Every image needs a matching label file; class distribution should be roughly balanced. Augmentation (rotation, scale, perspective, HSV jitter, mosaic) is applied on the fly by Ultralytics during training, so we don't pre-generate transformed copies.

In [3]:
import yaml

names = yaml.safe_load(DATA_YAML.read_text())["names"]

for split in ("train", "val"):
    images = sorted(p for p in (YOLO_DATASET / "images" / split).glob("*") if p.is_file())
    labels = {p.stem: p for p in (YOLO_DATASET / "labels" / split).glob("*.txt")}
    missing = [p.name for p in images if p.stem not in labels]
    print(f"{split}: {len(images)} images, {len(labels)} labels"
          + (f", MISSING labels for {missing}" if missing else ""))
    counts = Counter(int(p.read_text().split()[0]) for p in labels.values())
    for cls_index in sorted(counts):
        print(f"    {names[cls_index]:<10} {counts[cls_index]}")

train: 281 images, 281 labels


    11_1       23
    12_2       25
    13_3       32
    14_4       33
    15_5       29
    16_6       26
    17_7       41
    18_8       26
    19_9       28
    40_stop    18
val: 70 images, 70 labels


    11_1       6
    12_2       7
    13_3       7
    14_4       8
    15_5       7
    16_6       7
    17_7       10
    18_8       7
    19_9       7
    40_stop    4


In [4]:
# spot-check auto-generated boxes (green = card outline, orange = fallback)
review = sorted((YOLO_DATASET / "review" / "train").glob("*.jpg"))
sample = random.sample(review, min(6, len(review)))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax in axes.flat:
    ax.axis("off")
for ax, path in zip(axes.flat, sample):
    ax.imshow(cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB))
    ax.set_title(path.name, fontsize=8)
plt.tight_layout()
plt.show()

In [5]:
model = YOLO("yolo26n.pt")  # detection task, pretrained on COCO

# heavier on-the-fly augmentation to stretch the small dataset: every epoch the
# model sees a freshly rotated/shifted/scaled/sheared/color-jittered version of
# each capture, so the effective number of distinct training examples scales
# with epochs. fliplr stays 0: mirroring turns digits and the left/right
# arrow cards into the wrong class.
#
# The whole network trains (no freeze): COCO backbone features can't separate
# printed digits, so the backbone must adapt.
results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=16,
    degrees=20,       # rotation
    translate=0.2,    # shift up to 20% of the frame
    scale=0.6,        # zoom in/out
    shear=10,         # shear angle
    perspective=0.0005,
    hsv_v=0.5,        # brightness jitter (lighting varies on the robot)
    mixup=0.15,       # blend pairs of images
    fliplr=0.0,       # NO horizontal flip (see above)
    device=device,
    seed=42,
    project=ROOT / "runs",   # keep runs in this repo regardless of global yolo settings
)
BEST = Path(results.save_dir) / "weights" / "best.pt"
BEST

New https://pypi.org/project/ultralytics/8.4.154 available 😃 Update with 'pip install -U ultralytics'


Ultralytics 8.4.153 🚀 Python-3.11.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 4000 Blackwell, 23986MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/image-rec/dataset_yolo/data.yaml, degrees=20, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-6, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=

Overriding model.yaml nc=80 with nc=30



                   from  n    params  module                                       arguments                     


...

                  19_9          7          7      0.975          1      0.995       0.87


               40_stop          4          4      0.954          1      0.995      0.928


Speed: 0.1ms preprocess, 77.1ms inference, 0.0ms loss, 0.6ms postprocess per image


Results saved to /workspace/image-rec/runs/train-6


PosixPath('/workspace/image-rec/runs/train-6/weights/best.pt')

### Evaluate and demo

Validation metrics on the held-out split, then a visual check: run the trained detector on sample images and draw its predicted boxes.

In [6]:
model = YOLO(BEST)
metrics = model.val(data=DATA_YAML, device=device, project=ROOT / "runs")
print(f"mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}")

Ultralytics 8.4.153 🚀 Python-3.11.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 4000 Blackwell, 23986MiB)


YOLO26n summary (fused): 120 layers, 2,380,686 parameters, 0 gradients, 5.3 GFLOPs


WARNING ⚠️ val: Slow image access detected (ping: 1.4±0.4 ms, read: 13.1±4.7 MB/s, size: 89.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips


val: Scanning /workspace/image-rec/dataset_yolo/labels/val.cache... 70 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 70/70 21.0Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 20% ━━────────── 1/5 2.3s/it 0.7s<9.0s

...

                  19_9          7          7      0.975          1      0.995      0.868


               40_stop          4          4      0.953          1      0.995      0.928


Speed: 4.7ms preprocess, 3.4ms inference, 0.0ms loss, 0.9ms postprocess per image


Results saved to /workspace/image-rec/runs/val-4


mAP50: 0.964   mAP50-95: 0.849


In [7]:
# predicted bounding boxes on sample validation images (falls back to train if val is empty)
val_images = sorted((YOLO_DATASET / "images" / "val").glob("*")) or sorted(
    (YOLO_DATASET / "images" / "train").glob("*")
)
sample = random.sample(val_images, min(6, len(val_images)))
predictions = model.predict([str(p) for p in sample], imgsz=640, device=device, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax in axes.flat:
    ax.axis("off")
for ax, res in zip(axes.flat, predictions):
    ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
    detected = [model.names[int(c)] for c in res.boxes.cls]
    ax.set_title(", ".join(detected) or "no detection", fontsize=9)
plt.tight_layout()
plt.show()

In [8]:
# snapshot the deployment weights for server.py
# prediction class -> image ID for the task: image_id = int(model.names[cls].split("_")[0])
import shutil

deployed = ROOT / "weights" / "best.pt"
deployed.parent.mkdir(exist_ok=True)
shutil.copy2(BEST, deployed)
print(f"copied {BEST} -> {deployed}")

copied /workspace/image-rec/runs/train-6/weights/best.pt -> /workspace/image-rec/weights/best.pt
